# Tutorial 1: Working with ENDF (Evaluated Nuclear Data File) Data

## Overview

ENDF is the primary evaluated nuclear data library used in the United States. It contains comprehensive, validated nuclear data that has been carefully evaluated by experts.

### What you'll learn:
- How to install and use OpenMC for ENDF data access
- How to download nuclear data files
- How to extract and plot neutron cross-sections
- Understanding different reaction types (MT numbers)

### Prerequisites:
```bash
pip install openmc matplotlib numpy
```

## 1. Setting Up OpenMC

OpenMC is a Monte Carlo particle transport code that includes excellent tools for working with nuclear data.

In [ ]:
import openmc
import openmc.data
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path

print(f"OpenMC version: {openmc.__version__}")

## 2. Downloading ENDF Data

OpenMC can automatically download ENDF/B-VIII.0 data from NNDC (National Nuclear Data Center).

For this tutorial, we'll download data for a few isotopes. In practice, you might want to download the full library.

In [ ]:
# Define data directory
data_dir = Path('../data/endf')
data_dir.mkdir(parents=True, exist_ok=True)

# We'll work with U-235 as an example
# ENDF data can be downloaded from NNDC
print("To download ENDF data, you can use:")
print("openmc.data.download_endf_chain()")
print("\nFor this tutorial, we'll use openmc's built-in data access methods")

## 3. Loading Nuclear Data

We can load individual isotope data or use OpenMC's data library.

In [ ]:
# Method 1: Download specific ENDF file
# This downloads the ENDF file for U-235 from NNDC

import urllib.request
import os

# U-235 ENDF file (ENDF/B-VIII.0)
url = 'https://www.nndc.bnl.gov/endf-b8.0/zips/ENDF-B-VIII.0_neutrons_n-092_U_235.zip'
zip_file = data_dir / 'u235.zip'

if not zip_file.exists():
    print(f"Downloading U-235 ENDF data...")
    urllib.request.urlretrieve(url, zip_file)
    
    # Extract the zip file
    import zipfile
    with zipfile.ZipFile(zip_file, 'r') as zip_ref:
        zip_ref.extractall(data_dir)
    print("Download complete!")
else:
    print("U-235 data already downloaded")

In [ ]:
# Find the extracted ENDF file
endf_file = None
for file in data_dir.rglob('n-092_U_235.endf'):
    endf_file = file
    break

if endf_file:
    print(f"Found ENDF file: {endf_file}")
    # Read the ENDF file using OpenMC
    u235 = openmc.data.IncidentNeutron.from_endf(endf_file)
    print(f"\nLoaded: {u235.name}")
    print(f"Atomic mass: {u235.atomic_weight}")
    print(f"Available reactions: {len(u235.reactions)} reactions")
else:
    print("ENDF file not found. Using alternative method...")
    # Alternative: use openmc's data library if available
    print("You may need to set up OPENMC_CROSS_SECTIONS environment variable")

## 4. Understanding MT Numbers

ENDF uses MT (Material/Reaction Type) numbers to identify different reactions:

- **MT=1**: Total cross-section
- **MT=2**: Elastic scattering
- **MT=18**: Fission
- **MT=102**: (n,γ) radiative capture
- **MT=4**: Inelastic scattering

Let's explore what reactions are available:

In [ ]:
if 'u235' in locals():
    print("Available reactions (MT numbers):")
    print("-" * 50)
    for mt in sorted(u235.reactions.keys())[:20]:  # Show first 20
        reaction = u235[mt]
        print(f"MT={mt:3d}: {reaction}")

## 5. Extracting and Plotting Cross-Section Data

Now let's extract and visualize some key cross-sections for U-235.

In [ ]:
if 'u235' in locals():
    # Define energy grid (in eV)
    energy = np.logspace(-2, 7, 10000)  # 0.01 eV to 10 MeV
    
    # Extract cross-sections for different reactions
    reactions_to_plot = {
        1: 'Total',
        2: 'Elastic',
        18: 'Fission',
        102: 'Capture (n,γ)'
    }
    
    plt.figure(figsize=(14, 8))
    
    for mt, label in reactions_to_plot.items():
        if mt in u235.reactions:
            # Get the cross-section values
            xs = u235[mt].xs['0K'](energy)
            plt.loglog(energy, xs, label=f'{label} (MT={mt})', linewidth=2)
    
    plt.xlabel('Neutron Energy (eV)', fontsize=14)
    plt.ylabel('Cross-section (barns)', fontsize=14)
    plt.title('U-235 Neutron Cross-Sections (ENDF/B-VIII.0)', fontsize=16)
    plt.legend(fontsize=12)
    plt.grid(True, alpha=0.3, which='both')
    plt.xlim(1e-2, 1e7)
    plt.tight_layout()
    plt.savefig(data_dir / 'u235_cross_sections.png', dpi=150)
    plt.show()
    
    print("Plot saved to:", data_dir / 'u235_cross_sections.png')

## 6. Zooming into the Resonance Region

The resonance region (typically 1 eV - 10 keV) shows interesting structure where cross-sections vary dramatically.

In [ ]:
if 'u235' in locals():
    # Focus on resonance region
    energy_res = np.logspace(0, 4, 50000)  # 1 eV to 10 keV, high resolution
    
    plt.figure(figsize=(14, 8))
    
    # Plot fission and capture
    if 18 in u235.reactions:
        xs_fission = u235[18].xs['0K'](energy_res)
        plt.loglog(energy_res, xs_fission, label='Fission', linewidth=1.5, alpha=0.8)
    
    if 102 in u235.reactions:
        xs_capture = u235[102].xs['0K'](energy_res)
        plt.loglog(energy_res, xs_capture, label='Capture', linewidth=1.5, alpha=0.8)
    
    plt.xlabel('Neutron Energy (eV)', fontsize=14)
    plt.ylabel('Cross-section (barns)', fontsize=14)
    plt.title('U-235 Resonance Region Detail', fontsize=16)
    plt.legend(fontsize=12)
    plt.grid(True, alpha=0.3, which='both')
    plt.xlim(1, 1e4)
    plt.tight_layout()
    plt.show()

## 7. Comparing Multiple Isotopes

Let's download and compare different isotopes. We'll compare U-235 and U-238.

In [ ]:
# Download U-238 data
url_u238 = 'https://www.nndc.bnl.gov/endf-b8.0/zips/ENDF-B-VIII.0_neutrons_n-092_U_238.zip'
zip_file_u238 = data_dir / 'u238.zip'

if not zip_file_u238.exists():
    print("Downloading U-238 ENDF data...")
    urllib.request.urlretrieve(url_u238, zip_file_u238)
    
    with zipfile.ZipFile(zip_file_u238, 'r') as zip_ref:
        zip_ref.extractall(data_dir)
    print("Download complete!")

# Find and load U-238
endf_file_u238 = None
for file in data_dir.rglob('n-092_U_238.endf'):
    endf_file_u238 = file
    break

if endf_file_u238:
    u238 = openmc.data.IncidentNeutron.from_endf(endf_file_u238)
    print(f"Loaded: {u238.name}")

In [ ]:
# Compare fission cross-sections
if 'u235' in locals() and 'u238' in locals():
    energy = np.logspace(-2, 7, 10000)
    
    plt.figure(figsize=(14, 8))
    
    # U-235 fission
    if 18 in u235.reactions:
        xs_235 = u235[18].xs['0K'](energy)
        plt.loglog(energy, xs_235, label='U-235 Fission', linewidth=2)
    
    # U-238 fission (threshold reaction)
    if 18 in u238.reactions:
        xs_238 = u238[18].xs['0K'](energy)
        plt.loglog(energy, xs_238, label='U-238 Fission', linewidth=2)
    
    plt.xlabel('Neutron Energy (eV)', fontsize=14)
    plt.ylabel('Fission Cross-section (barns)', fontsize=14)
    plt.title('Comparison: U-235 vs U-238 Fission Cross-Sections', fontsize=16)
    plt.legend(fontsize=12)
    plt.grid(True, alpha=0.3, which='both')
    plt.tight_layout()
    plt.show()
    
    print("\nNote: U-235 is fissile (fissions with thermal neutrons)")
    print("      U-238 is fissionable (requires fast neutrons, threshold ~1 MeV)")

## 8. Extracting Data for Machine Learning

Let's extract the cross-section data into a format suitable for machine learning analysis.

In [ ]:
if 'u235' in locals():
    import pandas as pd
    
    # Create a DataFrame with cross-section data
    energy = np.logspace(-2, 7, 10000)
    
    data = {'Energy_eV': energy}
    
    # Add cross-sections for available reactions
    for mt in [1, 2, 18, 102]:
        if mt in u235.reactions:
            data[f'MT{mt}_barns'] = u235[mt].xs['0K'](energy)
    
    df = pd.DataFrame(data)
    
    # Save to CSV
    csv_file = data_dir / 'u235_cross_sections.csv'
    df.to_csv(csv_file, index=False)
    
    print(f"Data saved to: {csv_file}")
    print(f"\nDataFrame shape: {df.shape}")
    print("\nFirst few rows:")
    print(df.head())
    
    print("\nStatistics:")
    print(df.describe())

## 9. Key Takeaways

1. **ENDF data is evaluated and validated** - It represents the best available nuclear data
2. **MT numbers** identify different reaction types
3. **Energy dependence** is crucial - cross-sections vary by many orders of magnitude
4. **Resonance structure** provides detailed information about nuclear physics
5. **OpenMC** provides excellent tools for accessing and manipulating ENDF data

## Next Steps

- Explore other isotopes (Fe-56, H-1, Pu-239, etc.)
- Compare ENDF with experimental data (see Tutorial 2: EXFOR)
- Use this data for machine learning applications
- Investigate uncertainties in evaluated data

## Resources

- ENDF Database: https://www.nndc.bnl.gov/endf/
- OpenMC Documentation: https://docs.openmc.org/
- ENDF Format Manual: https://www.nndc.bnl.gov/endfdocs/